In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../data/customer_transactions.csv")
display(df.head())

,date,store_nbr,lylty_card_nbr,txn_id,prod_nbr,prod_qty,tot_sales,clean_prod_name,net_weight,brand,lifestage,premium_customer
0,2018-10-17,1,1000,1,5,2,6.0,natural chip compny seasalt 175 g,175,Natural Chip Company,YOUNG SINGLES/COUPLES,Premium
1,2019-05-14,1,1307,348,66,3,6.3,ccs nacho cheese 175 g,175,CCs,MIDAGE SINGLES/COUPLES,Budget
2,2019-05-20,1,1343,383,61,2,2.9,smiths crinkle cut chips chicken 170 g,170,Smith's,MIDAGE SINGLES/COUPLES,Budget
3,2018-08-17,2,2373,974,69,5,15.0,smiths chip thinly s cream onion 175 g,175,Smith's,MIDAGE SINGLES/COUPLES,Budget
4,2018-08-18,2,2426,1038,108,3,13.8,kettle tortilla chpshny jlpno chili 150 g,150,Kettle,MIDAGE SINGLES/COUPLES,Budget


In [2]:
brand_sales = df.groupby("brand")["tot_sales"].sum()

top_brands = (
    brand_sales.nlargest(10)
    .reset_index(name="total_sales")
)
top_brands["sales_share_%"] = top_brands["total_sales"] / brand_sales.sum() * 100

# tidy up the format to show 2 decimal places and a percentage sign
top_brands["sales_share_%"] = top_brands["sales_share_%"].map(lambda v: f"{v:.2f}%")

print(top_brands)

           brand  total_sales sales_share_%
0         Kettle     390239.8        21.70%
1        Doritos     226329.9        12.59%
2        Smith's     210440.0        11.70%
3       Pringles     177655.5         9.88%
4      Infuzions      99047.6         5.51%
5          Thins      88852.5         4.94%
6  Red Rock Deli      87607.5         4.87%
7       Twisties      81522.1         4.53%
8       Tostitos      79789.6         4.44%
9           Cobs      70569.8         3.92%


**Kettle is the standount brand** <br>
- accounting for a fifth of all sales, the strongest performance out of 20 brands<br>
<br>
**Dorito's, Smith's and Pringles are reliable, accounting for approx. 10-13% each** <br>
<br>
- these are the most well-received brands globally. this is a great starting point and would help isolate top brands <br>
- positioning items from these brands on featured sections, and near the checkout isle would take advantage on their widespread appeal. <br>

*let's dig deeper on each cohort's top brands*

In [10]:
# calculate brand affinity for OLDER SINGLES/COUPLES segment

segment_older_sc = df[
    df["lifestage"] == "OLDER SINGLES/COUPLES"
]

other_older_lifestages = ["OLDER FAMILIES", "RETIREES"]

other_older = df[
    df["lifestage"].isin(other_older_lifestages)
]

brand_affinity_oldersc = pd.merge(
    (
        segment_older_sc.groupby("brand")["prod_qty"].sum()
        / segment_older_sc["prod_qty"].sum()
    ).rename("target_segment"),
    (
        other_older.groupby("brand")["prod_qty"].sum()
        / other_older["prod_qty"].sum()
    ).rename("other"),
    on="brand"
)
#brand affinity compared to the rest of the population

brand_affinity_oldersc["affinity_to_brand"] = (
    brand_affinity_oldersc["target_segment"]
    / brand_affinity_oldersc["other"]
)

brand_affinity_older = brand_affinity_oldersc.sort_values(
    "affinity_to_brand",
    ascending=False
)

display(brand_affinity_older)

,target_segment,other,affinity_to_brand
brand,,,
Tostitos,0.040578,0.037504,1.081964
Kettle,0.176223,0.165548,1.064480
Doritos,0.106247,0.101303,1.048807
Cobs,0.040350,0.038717,1.042186
Pringles,0.105202,0.101257,1.038956
Tyrrells,0.026645,0.025826,1.031684
Infuzions,0.058641,0.057248,1.024328
Thins,0.059241,0.057940,1.022451
Cheezels,0.018798,0.018417,1.020691


- Older S/C are the largest population, the most consistent across 3 membership tiers, and acoount for 20% of sales, the table above shows that they do not exhibit extreme liking or disliking towards a brand <br>

        - this is backed by their affinity scores hovering around 1, with 0.88 as the lowest.
        - 0.88 = they are 12% less likely to buy chips from the Burger Rings brand
        - affinity is dispersed since this is a culmination of 3 membership tiers
        - this cohort is open to variety in brands

**Older S/C produce high sales but does not need targetting interventions due to their balanced choices**

In [4]:
segment_b_olderfamilies = df[
    (df["lifestage"] == "OLDER FAMILIES") &
    (df["premium_customer"] == "Budget")
]

other_older_lifestages = ["OLDER SINGLES/COUPLES", "RETIREES"]

other_older = df[
    df["lifestage"].isin(other_older_lifestages)
]

brand_affinity_olderfamilies = pd.merge(
    (
        segment_b_olderfamilies.groupby("brand")["prod_qty"].sum()
        / segment_b_olderfamilies["prod_qty"].sum()
    ).rename("target_segment"),
    (
        other_older.groupby("brand")["prod_qty"].sum()
        / other_older["prod_qty"].sum()
    ).rename("other"),
    on="brand"
)
#brand affinity compared to the rest of the population

brand_affinity_olderfamilies["affinity_to_brand"] = (
    brand_affinity_olderfamilies["target_segment"]
    / brand_affinity_olderfamilies["other"]
)

brand_affinity_olderfamilies = brand_affinity_olderfamilies.sort_values(
    "affinity_to_brand",
    ascending=False
)

display(brand_affinity_olderfamilies)


,target_segment,other,affinity_to_brand
brand,,,
Red Rock Deli,0.079744,0.058933,1.353119
Woolworths,0.056668,0.043351,1.307180
CCs,0.021223,0.016348,1.298193
French Fries,0.006641,0.005134,1.293511
Burger Rings,0.007243,0.005678,1.275684
Natural Chip Company,0.034674,0.027997,1.238502
Sunbites,0.014197,0.011480,1.236646
Cheetos,0.013186,0.010986,1.200307
Smith's,0.130468,0.111771,1.167283


Budget Older Families are 35% more likely to buy chips from Red Rock Deli compared to other brands

Seven other brands are highly preferred by this cohort, with affinity scores >= 1.20. 

In [5]:
segment_main_retirees = df[
    (df["lifestage"] == "RETIREES") &
    (df["premium_customer"] == "Mainstream")
]

other_older_lifestages = ["OLDER SINGLES/COUPLES", "OLDER FAMILIES"]

other_older = df[
    df["lifestage"].isin(other_older_lifestages)
]

brand_affinity_retirees = pd.merge(
    (
        segment_main_retirees.groupby("brand")["prod_qty"].sum()
        / segment_main_retirees["prod_qty"].sum()
    ).rename("target_segment"),
    (
        other_older.groupby("brand")["prod_qty"].sum()
        / other_older["prod_qty"].sum()
    ).rename("other"),
    on="brand"
)
#brand affinity compared to the rest of the population

brand_affinity_retirees["affinity_to_brand"] = (
    brand_affinity_retirees["target_segment"]
    / brand_affinity_retirees["other"]
)

brand_affinity_retirees = brand_affinity_retirees.sort_values(
    "affinity_to_brand",
    ascending=False
)

display(brand_affinity_retirees)


,target_segment,other,affinity_to_brand
brand,,,
Twisties,0.040440,0.037953,1.065518
Pringles,0.106567,0.100191,1.063639
Grain Waves,0.033557,0.031641,1.060564
Thins,0.060793,0.057324,1.060518
Doritos,0.106007,0.100414,1.055699
Infuzions,0.059913,0.057144,1.048443
Kettle,0.171468,0.165224,1.037791
Cheezels,0.019206,0.018533,1.036298
Cobs,0.039693,0.038399,1.033696


**Mainstream Retirees** show dispersed affinity scores, exhibiting no strong preference towards brands {similar to Older Families}

*Slight preference at best*
- Approximately 6-6.5% brand affinity on its top 4 brands [Twisties,Pringles,Grainwaves,Thins]

In [6]:
segment_b_youngfamilies = df[
    (df["lifestage"] == "YOUNG FAMILIES") &
    (df["premium_customer"] == "Budget")
]

other_young_lifestages = ["YOUNG SINGLES/COUPLES"]

other_young = df[
    df["lifestage"].isin(other_young_lifestages)
]

brand_affinity_youngfamilies = pd.merge(
    (
        segment_b_youngfamilies.groupby("brand")["prod_qty"].sum()
        / segment_b_youngfamilies["prod_qty"].sum()
    ).rename("target_segment"),
    (
        other_young.groupby("brand")["prod_qty"].sum()
        / other_young["prod_qty"].sum()
    ).rename("other"),
    on="brand"
)
#brand affinity compared to the rest of the population

brand_affinity_youngfamilies["affinity_to_brand"] = (
    brand_affinity_youngfamilies["target_segment"]
    / brand_affinity_youngfamilies["other"]
)

brand_affinity_youngfamilies = brand_affinity_youngfamilies.sort_values(
    "affinity_to_brand",
    ascending=False
)

display(brand_affinity_youngfamilies)


,target_segment,other,affinity_to_brand
brand,,,
Sunbites,0.014534,0.010084,1.441351
Burger Rings,0.007034,0.005050,1.392814
Woolworths,0.055510,0.041174,1.348188
Red Rock Deli,0.076669,0.057179,1.340871
CCs,0.022414,0.017005,1.318078
Natural Chip Company,0.034468,0.026427,1.304237
French Fries,0.007063,0.005469,1.291327
Cheetos,0.012345,0.010245,1.205000
Smith's,0.126138,0.107630,1.171964


**Budget Young Families** exhibit strong preference with **Sunbites**, with a whopping *44% brand affinity* <br>

Brands from top 2-6 spots range from 30-39% positive brand affinity as well; the strongest brand preference signal by far. 

Despite this, the bottom 5 are still only negative 10-20% less preferred (typical; even similar to other cohorts)

Let's compare this to Older Families


In [7]:
#looking at the affinity scores side-by-side

brand_affinity_comparefamilies = pd.merge(
    brand_affinity_youngfamilies[["affinity_to_brand"]]
        .rename(columns={"affinity_to_brand": "youngfamilies_affinity"}),
    brand_affinity_olderfamilies[["affinity_to_brand"]]
        .rename(columns={"affinity_to_brand": "olderfamilies_affinity"}),
    left_index=True,
    right_index=True,
    how="outer"
).reset_index()

brand_affinity_comparefamilies = brand_affinity_comparefamilies.sort_values(
    "youngfamilies_affinity",
    ascending=False
)

display(brand_affinity_comparefamilies)

,brand,youngfamilies_affinity,olderfamilies_affinity
14,Sunbites,1.441351,1.236646
0,Burger Rings,1.392814,1.275684
19,Woolworths,1.348188,1.307180
12,Red Rock Deli,1.340871,1.353119
1,CCs,1.318078,1.298193
10,Natural Chip Company,1.304237,1.238502
6,French Fries,1.291327,1.293511
2,Cheetos,1.205000,1.200307
13,Smith's,1.171964,1.167283
3,Cheezels,1.023198,1.083413


In [8]:
# Correlation between the two cohorts' brand affinity scores
affinity_cols = ["youngfamilies_affinity", "olderfamilies_affinity"]

pearson_corr = brand_affinity_comparefamilies[affinity_cols].corr(method="pearson").iloc[0, 1]
spearman_corr = brand_affinity_comparefamilies[affinity_cols].corr(method="spearman").iloc[0, 1]

n_brands = brand_affinity_comparefamilies[affinity_cols].dropna().shape[0]

print(f"Pearson correlation:  {pearson_corr:.3f}")
print(f"Spearman correlation: {spearman_corr:.3f}")
print(f"Brands compared:      {n_brands}")

Pearson correlation:  0.954
Spearman correlation: 0.875
Brands compared:      20


**Young and older families** show a *strong overlap in brand preferences* (Pearson 0.95, Spearman 0.88), so the brands that over-index with one cohort largely over-index with the other. The slightly lower rank correlation suggests the order of preference shifts between the two groups, even though the overall favorites are shared.

In [9]:
segment_main_youngsc = df[
    (df["lifestage"] == "YOUNG SINGLES/COUPLES") &
    (df["premium_customer"] == "Mainstream")
]

other_young_lifestages = ["YOUNG FAMILIES"]

other_young = df[
    df["lifestage"].isin(other_young_lifestages)
]

brand_affinity_youngsc = pd.merge(
    (
        segment_main_youngsc.groupby("brand")["prod_qty"].sum()
        / segment_main_youngsc["prod_qty"].sum()
    ).rename("target_segment"),
    (
        other_young.groupby("brand")["prod_qty"].sum()
        / other_young["prod_qty"].sum()
    ).rename("other"),
    on="brand"
)
#brand affinity compared to the rest of the population

brand_affinity_youngsc["affinity_to_brand"] = (
    brand_affinity_youngsc["target_segment"]
    / brand_affinity_youngsc["other"]
)

brand_affinity_youngsc = brand_affinity_youngsc.sort_values(
    "affinity_to_brand",
    ascending=False
)

display(brand_affinity_youngsc)


,target_segment,other,affinity_to_brand
brand,,,
Twisties,0.046350,0.035184,1.317343
Tyrrells,0.031666,0.024767,1.278551
Kettle,0.198698,0.155716,1.276029
Doritos,0.123203,0.096687,1.274248
Pringles,0.119850,0.095264,1.258083
Tostitos,0.045574,0.036350,1.253748
Cobs,0.044798,0.037568,1.192476
Infuzions,0.064912,0.054916,1.182017
Thins,0.060590,0.054045,1.121104


brand targetting is a lot more crucial for Mainstream Young Singles/Couples

- strong affinity with its top 6 brands
- strong replusion towards the bottom 8 brands [affinity = 0.58 to 0.40(lowest)]

*let's see how these choices compare to other cohorts'*

In [11]:
#looking at the affinity scores side-by-side

brand_affinity_comparesc = pd.merge(
    brand_affinity_youngsc[["affinity_to_brand"]]
        .rename(columns={"affinity_to_brand": "youngsc_affinity"}),
    brand_affinity_oldersc[["affinity_to_brand"]]
        .rename(columns={"affinity_to_brand": "oldersc_affinity"}),
    left_index=True,
    right_index=True,
    how="outer"
).reset_index()

brand_affinity_comparesc = brand_affinity_comparesc.sort_values(
    "youngsc_affinity",
    ascending=False
)

display(brand_affinity_comparesc)

,brand,youngsc_affinity,oldersc_affinity
17,Twisties,1.317343,1.002568
18,Tyrrells,1.278551,1.031684
9,Kettle,1.276029,1.064480
5,Doritos,1.274248,1.048807
11,Pringles,1.258083,1.038956
16,Tostitos,1.253748,1.081964
4,Cobs,1.192476,1.042186
8,Infuzions,1.182017,1.024328
15,Thins,1.121104,1.022451
7,Grain Waves,1.083405,1.002089


In [12]:
# Looking at the affinity scores side-by-side (young SC, older SC, young families)

brand_affinity_comparesc = (
    brand_affinity_youngsc[["affinity_to_brand"]]
        .rename(columns={"affinity_to_brand": "youngsc_affinity"})
    .merge(
        brand_affinity_oldersc[["affinity_to_brand"]]
            .rename(columns={"affinity_to_brand": "oldersc_affinity"}),
        left_index=True, right_index=True, how="outer"
    )
    .merge(
        brand_affinity_youngfamilies[["affinity_to_brand"]]
            .rename(columns={"affinity_to_brand": "youngfamilies_affinity"}),
        left_index=True, right_index=True, how="outer"
    )
    .reset_index()
)

brand_affinity_comparesc = brand_affinity_comparesc.sort_values(
    "youngsc_affinity",
    ascending=False
)

display(brand_affinity_comparesc)

,brand,youngsc_affinity,oldersc_affinity,youngfamilies_affinity
17,Twisties,1.317343,1.002568,0.840158
18,Tyrrells,1.278551,1.031684,0.885502
9,Kettle,1.276029,1.064480,0.876735
5,Doritos,1.274248,1.048807,0.914962
11,Pringles,1.258083,1.038956,0.862791
16,Tostitos,1.253748,1.081964,0.903749
4,Cobs,1.192476,1.042186,0.827316
8,Infuzions,1.182017,1.024328,0.932938
15,Thins,1.121104,1.022451,0.914870
7,Grain Waves,1.083405,1.002089,1.004856


despite score fluctuations, preferences of both Singles/Couples lifestages are similar. <df>

lifestage is a stronger indicator of preference, and not age. <df>

Young Singles/Couples share favourites with Older Singles/Couples, but have oppposite taste compared to Young Families.  <df>

*let's verify this hypothesis*

In [ ]:
# compare preferences using a correlation matrix

affinity_cols = ["youngsc_affinity", "oldersc_affinity", "youngfamilies_affinity"]
display(brand_affinity_comparesc[affinity_cols].corr(method="pearson"))
display(brand_affinity_comparesc[affinity_cols].corr(method="spearman"))

,youngsc_affinity,oldersc_affinity,youngfamilies_affinity
youngsc_affinity,1.000000,0.917963,-0.983955
oldersc_affinity,0.917963,1.000000,-0.917233
youngfamilies_affinity,-0.983955,-0.917233,1.000000


,youngsc_affinity,oldersc_affinity,youngfamilies_affinity
youngsc_affinity,1.000000,0.87218,-0.941353
oldersc_affinity,0.872180,1.00000,-0.872180
youngfamilies_affinity,-0.941353,-0.87218,1.000000


the correlations support the hypothesis. family-based cohorts have similar brand preferences, and lifestage is better than age as an indicator of brand affinity. 

In [17]:
#looking at the affinity scores side-by-side

brand_affinity_compareolder_neutral = pd.merge(
    brand_affinity_retirees[["affinity_to_brand"]]
        .rename(columns={"affinity_to_brand": "retirees_affinity"}),
    brand_affinity_oldersc[["affinity_to_brand"]]
        .rename(columns={"affinity_to_brand": "oldersc_affinity"}),
    left_index=True,
    right_index=True,
    how="outer"
).reset_index()

brand_affinity_compareolder_neutral = brand_affinity_compareolder_neutral.sort_values(
    "retirees_affinity",
    ascending=False
)

display(brand_affinity_compareolder_neutral)

,brand,retirees_affinity,oldersc_affinity
17,Twisties,1.065518,1.002568
11,Pringles,1.063639,1.038956
7,Grain Waves,1.060564,1.002089
15,Thins,1.060518,1.022451
5,Doritos,1.055699,1.048807
8,Infuzions,1.048443,1.024328
9,Kettle,1.037791,1.064480
3,Cheezels,1.036298,1.020691
4,Cobs,1.033696,1.042186
18,Tyrrells,1.012884,1.031684


In [18]:
# Correlation between the two cohorts' brand affinity scores
affinity_cols = ["retirees_affinity", "oldersc_affinity"]

pearson_corr = brand_affinity_compareolder_neutral[affinity_cols].corr(method="pearson").iloc[0, 1]
spearman_corr = brand_affinity_compareolder_neutral[affinity_cols].corr(method="spearman").iloc[0, 1]

n_brands = brand_affinity_compareolder_neutral[affinity_cols].dropna().shape[0]

print(f"Pearson correlation:  {pearson_corr:.3f}")
print(f"Spearman correlation: {spearman_corr:.3f}")
print(f"Brands compared:      {n_brands}")

Pearson correlation:  0.742
Spearman correlation: 0.632
Brands compared:      20


In [15]:
# Looking at the affinity scores side-by-side (retirees, older families, older SC)

brand_affinity_compareolder = (
    brand_affinity_retirees[["affinity_to_brand"]]
        .rename(columns={"affinity_to_brand": "retirees_affinity"})
    .merge(
        brand_affinity_olderfamilies[["affinity_to_brand"]]
            .rename(columns={"affinity_to_brand": "olderfamilies_affinity"}),
        left_index=True, right_index=True, how="outer"
    )
    .merge(
        brand_affinity_oldersc[["affinity_to_brand"]]
            .rename(columns={"affinity_to_brand": "oldersc_affinity"}),
        left_index=True, right_index=True, how="outer"
    )
    .reset_index()
)

brand_affinity_compareolder = brand_affinity_compareolder.sort_values(
    "retirees_affinity",
    ascending=False
)

affinity_cols = ["retirees_affinity", "olderfamilies_affinity", "oldersc_affinity"]

# Output 1: side-by-side tally of affinity scores
display(brand_affinity_compareolder)

# Output 2: correlation matrix of the three cohorts
display(brand_affinity_compareolder[affinity_cols].corr(method="pearson").round(3))

,brand,retirees_affinity,olderfamilies_affinity,oldersc_affinity
17,Twisties,1.065518,0.959759,1.002568
11,Pringles,1.063639,0.873569,1.038956
7,Grain Waves,1.060564,0.982845,1.002089
15,Thins,1.060518,0.914767,1.022451
5,Doritos,1.055699,0.885819,1.048807
8,Infuzions,1.048443,0.942235,1.024328
9,Kettle,1.037791,0.882187,1.064480
3,Cheezels,1.036298,1.083413,1.020691
4,Cobs,1.033696,0.879278,1.042186
18,Tyrrells,1.012884,0.849580,1.031684


,retirees_affinity,olderfamilies_affinity,oldersc_affinity
retirees_affinity,1.000,-0.858,0.742
olderfamilies_affinity,-0.858,1.000,-0.926
oldersc_affinity,0.742,-0.926,1.000


In [19]:
import pandas as pd

# Brand affinity scores for the 5 cohorts, aligned on the brand index
brand_affinity_all = pd.concat(
    {
        "youngfamilies_affinity": brand_affinity_youngfamilies["affinity_to_brand"],
        "olderfamilies_affinity": brand_affinity_olderfamilies["affinity_to_brand"],
        "youngsc_affinity":       brand_affinity_youngsc["affinity_to_brand"],
        "oldersc_affinity":       brand_affinity_oldersc["affinity_to_brand"],
        "retirees_affinity":      brand_affinity_retirees["affinity_to_brand"],
    },
    axis=1,
    join="outer",
)

# Correlation matrices
print("Pearson correlation:")
display(brand_affinity_all.corr(method="pearson").round(3))

print("Spearman correlation:")
display(brand_affinity_all.corr(method="spearman").round(3))

display(brand_affinity_all.notna().astype(int).T @ brand_affinity_all.notna().astype(int))

Pearson correlation:


,youngfamilies_affinity,olderfamilies_affinity,youngsc_affinity,oldersc_affinity,retirees_affinity
youngfamilies_affinity,1.000,0.954,-0.984,-0.917,-0.843
olderfamilies_affinity,0.954,1.000,-0.964,-0.926,-0.858
youngsc_affinity,-0.984,-0.964,1.000,0.918,0.847
oldersc_affinity,-0.917,-0.926,0.918,1.000,0.742
retirees_affinity,-0.843,-0.858,0.847,0.742,1.000


Spearman correlation:


,youngfamilies_affinity,olderfamilies_affinity,youngsc_affinity,oldersc_affinity,retirees_affinity
youngfamilies_affinity,1.000,0.875,-0.941,-0.872,-0.741
olderfamilies_affinity,0.875,1.000,-0.866,-0.919,-0.710
youngsc_affinity,-0.941,-0.866,1.000,0.872,0.747
oldersc_affinity,-0.872,-0.919,0.872,1.000,0.632
retirees_affinity,-0.741,-0.710,0.747,0.632,1.000


,youngfamilies_affinity,olderfamilies_affinity,youngsc_affinity,oldersc_affinity,retirees_affinity
youngfamilies_affinity,20,20,20,20,20
olderfamilies_affinity,20,20,20,20,20
youngsc_affinity,20,20,20,20,20
oldersc_affinity,20,20,20,20,20
retirees_affinity,20,20,20,20,20


across 20 brands, the following cohorts have shared preferences [and vice-versa] <br>

- strong brand affinity between Young Families AND Older Families <br>
- strong shared brand affinity between Young Singles/Couples AND Older Singles/Couples <br>
- moderately strong brand affinity betweem Young Singles/Couples AND Retirees <br>
- moderately strong brand affinity between Older Singles/Couples AND Retirees [correlation slightly less than Retirees == Young S/C] <br>

strong negative correlation

[Retirees, Older Singles/Couples, Young Singles/Couples] AND [Young Families, Older Families]